In [2]:
import pandas as pd
import os
import sys

src_path = os.path.abspath(os.path.join('.', 'src'))
sys.path.append(src_path)

from feature_engineering import calc_rolling_mean_w_shift

C:\Users\Leon\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv('meta\data\cleaned_merged_seasons.csv')

C:\Users\Leon\AppData\Local\Temp\ipykernel_15168\1876340079.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('meta\data\cleaned_merged_seasons.csv')


In [4]:
df.head()

,season_x,name,position,team_x,assists,bonus,bps,clean_sheets,creativity,element,...,team_h_score,threat,total_points,transfers_balance,transfers_in,transfers_out,value,was_home,yellow_cards,GW
0,2016-17,Aaron Cresswell,DEF,NaN,0,0,0,0,0.0,454,...,2.0,0.0,0,0,0,0,55,False,0,1
1,2016-17,Aaron Lennon,MID,NaN,0,0,6,0,0.3,142,...,1.0,0.0,1,0,0,0,60,True,0,1
2,2016-17,Aaron Ramsey,MID,NaN,0,0,5,0,4.9,16,...,3.0,23.0,2,0,0,0,80,True,0,1
3,2016-17,Abdoulaye Doucouré,MID,NaN,0,0,0,0,0.0,482,...,1.0,0.0,0,0,0,0,50,False,0,1
4,2016-17,Abdul Rahman Baba,DEF,NaN,0,0,0,0,0.0,80,...,2.0,0.0,0,0,0,0,55,True,0,1


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126076 entries, 0 to 126075
Data columns (total 37 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   season_x           126076 non-null  object 
 1   name               126076 non-null  object 
 2   position           126076 non-null  object 
 3   team_x             106042 non-null  object 
 4   assists            126076 non-null  int64  
 5   bonus              126076 non-null  int64  
 6   bps                126076 non-null  int64  
 7   clean_sheets       126076 non-null  int64  
 8   creativity         126076 non-null  float64
 9   element            126076 non-null  int64  
 10  fixture            126076 non-null  int64  
 11  goals_conceded     126076 non-null  int64  
 12  goals_scored       126076 non-null  int64  
 13  ict_index          126076 non-null  float64
 14  influence          126076 non-null  float64
 15  kickoff_time       126076 non-null  object 
 16  mi

In [25]:
df['round'] == df['GW']
df['id_player'] = df['name']

for col in df.select_dtypes(exclude=object).columns:
    df = calc_rolling_mean_w_shift(df, col, 3)
    df = calc_rolling_mean_w_shift(df, col, 5)
    df = calc_rolling_mean_w_shift(df, col, 7)

In [26]:
df_corr = df.corr(numeric_only=True)['total_points'].to_frame().sort_values(by='total_points', ascending=False)

In [27]:
df_corr.head(20)

,total_points
total_points,1.000000
bps,0.910427
influence,0.830978
ict_index,0.767776
bonus,0.751517
goals_scored,0.674469
minutes,0.639106
clean_sheets,0.580973
threat,0.565931
creativity,0.477327


In [28]:
def preprocess_df(df):
    cols_to_drop = [
        'season_x',
        'name',
        'position',
        'team_x',
        'assists',
        'bonus',
        'bps',
        'clean_sheets',
        'creativity',
        'element',
        'fixture',
        'goals_conceded',
        'goals_scored',
        'ict_index',
        'influence',
        'kickoff_time',
        'minutes',
        'opponent_team',
        'opp_team_name',
        'own_goals',
        'penalties_missed',
        'penalties_saved',
        'red_cards',
        'round',
        'saves',
        'selected',
        'team_a_score',
        'team_h_score',
        'threat',
        'transfers_balance',
        'transfers_in',
        'transfers_out',
        'value',
        'was_home',
        'yellow_cards',
        'GW',
        'id_player',
    ]

    df.drop(columns=cols_to_drop, inplace=True)
    df = df.select_dtypes(exclude=object)

    for col in df.columns:
        df[col] = df[col].fillna(df[col].mean())

    return df

In [29]:
df_test = df[df['season_x'] == '2023-24'].copy()
df_train = df[df['season_x'] != '2023-24'].copy()

df_train = preprocess_df(df_train)
df_test = preprocess_df(df_test)

In [30]:
X_train = df_train.drop('total_points', axis=1).values
y_train = df_train['total_points']
X_test = df_test.drop('total_points', axis=1).values
y_test = df_test['total_points']

In [35]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(
    f"RMSE: {rmse}\n"
    f"MAE: {mae}\n"
)

RMSE: 2.1160721804139184
MAE: 1.2287127515671754



In [34]:
df[df['rolling_mean_minutes_7'] == 0].shape[0] / df.shape[0]

0.20967511659633872

In [38]:
df_test = df[(df['season_x'] == '2023-24') & (df['rolling_mean_minutes_5'] != 0)].copy()
df_train = df[(df['season_x'] != '2023-24') & (df['rolling_mean_minutes_5'] != 0)].copy()

df_train = preprocess_df(df_train)
df_test = preprocess_df(df_test)

X_train = df_train.drop('total_points', axis=1).values
y_train = df_train['total_points']
X_test = df_test.drop('total_points', axis=1).values
y_test = df_test['total_points']

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(
    f"RMSE: {rmse}\n"
    f"MAE: {mae}\n"
)

RMSE: 2.5072036791502885
MAE: 1.6675925381046866



In [40]:
df_test['total_points_pred'] = y_pred
df_test[['total_points','total_points_pred']].head(10)

,total_points,total_points_pred
96579,0,1.657078
97217,0,1.622081
97838,0,1.485875
98529,0,1.184970
99235,0,0.930469
99952,0,0.757399
100701,0,0.686456
101480,0,0.436141
102208,0,0.471252
102939,0,0.480793
